In [ ]:
import numpy as np
import cv2
import glob
import os

PATTERN = (9, 6)  #coins INTERNES
DIM = 0.02     #un carré=2mm
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)


files = sorted(glob.glob('calib/*.jpg'))

#etape1
objp = np.zeros((PATTERN[0]*PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2) * DIM

objpoints, imgpoints = [], []
img_size = None
for fname in files:
    img = cv2.imread(fname)
    if img is None:
        print("illisible:", fname); continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if img_size is None:
        img_size = gray.shape[::-1]          # (w, h)
    elif gray.shape[::-1] != img_size:
        print("taille différente, ignorée:", fname, gray.shape[::-1]); continue
    ret, corners = cv2.findChessboardCorners(gray, PATTERN, None)
    print(fname, "mire détectée !" if ret else "mire NON détectée!!!!!")
    if ret:
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        objpoints.append(objp); imgpoints.append(corners2)

rms, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, img_size, None, None)
#calibration
h, w = gray.shape
rms, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, (w, h), None, None)
print("RMS:", rms, "\nK =\n", mtx, "\ndist =", dist.ravel())

#erreur de reprojection moyenne
tot = 0
for i in range(len(objpoints)):
    p2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
    e = cv2.norm(imgpoints[i], p2, cv2.NORM_L2) / len(p2)
    tot += e
    print(f"image {i}: {e:.4f} px")
print("erreur moyenne:", tot / len(objpoints))

#calcul focale
SENSOR_W, SENSOR_H =  6.0, 8.0   # car img 4:3
fovx, fovy, focal_mm, pp, aspect = cv2.calibrationMatrixValues(mtx, (w, h), SENSOR_W, SENSOR_H)
print(f"FOV: {fovx:.1f}° x {fovy:.1f}° | focale: {focal_mm:.3f} mm = {focal_mm/1000:.6f} m")
print("point principal (mm):", pp, "| aspect:", aspect)

np.savez('B.npz', mtx=mtx, dist=dist, rvecs=rvecs, tvecs=tvecs)

calib/IMG_0706.jpg mire détectée
calib/IMG_0708.jpg mire détectée
calib/IMG_0715.jpg mire détectée
calib/IMG_0716.jpg mire détectée
calib/IMG_0717.jpg mire détectée
calib/IMG_0718.jpg mire détectée
calib/IMG_0719.jpg mire détectée
calib/IMG_0720.jpg mire détectée
calib/IMG_0723.jpg mire détectée
calib/image.jpg mire détectée
calib/image2.jpg mire détectée
RMS: 2.5192694672538174 
K =
 [[4.35545895e+03 0.00000000e+00 2.13794669e+03]
 [0.00000000e+00 4.33704358e+03 2.87136476e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]] 
dist = [ 3.38063900e-01 -2.47728439e+00 -5.38790984e-04 -4.13666221e-05
  5.80381074e+00]
image 0: 0.6693 px
image 1: 0.5063 px
image 2: 0.2479 px
image 3: 0.1396 px
image 4: 0.1740 px
image 5: 0.3181 px
image 6: 0.3909 px
image 7: 0.2844 px
image 8: 0.3462 px
image 9: 0.0774 px
image 10: 0.1288 px
erreur moyenne: 0.2984576542963692
FOV: 52.4° x 66.7° | focale: 6.100 mm = 0.006100 m
point principal (mm): (2.9943230947162687, 4.02151927738543) | aspect: 0.9957718

In [19]:
#test undistort
img = cv2.imread('calib/IMG_0723.jpg')
h,  w = img.shape[:2]
newcameramtx, roi=cv2.getOptimalNewCameraMatrix(mtx,dist,(w,h),1,(w,h))

# undistort
dst = cv2.undistort(img, mtx, dist, None, newcameramtx)
cv2.imwrite('calibresultbeforeCrop.png',dst)
# crop the image
x,y,w,h = roi
dst = dst[y:y+h, x:x+w]
cv2.imwrite('calibresult.png',dst)

True

In [20]:
import numpy as np, cv2, glob
PATTERN, SQUARE = (9, 6), 0.02
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
with np.load('B.npz') as X:
    mtx, dist = X['mtx'], X['dist']

objp = np.zeros((PATTERN[0]*PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2) * SQUARE

L = 3 * SQUARE
axis = np.float32([[L,0,0],[0,L,0],[0,0,-L]]).reshape(-1,3)
cube = np.float32([[0,0,0],[0,L,0],[L,L,0],[L,0,0],
                   [0,0,-L],[0,L,-L],[L,L,-L],[L,0,-L]])

def draw_axes(img, corners, imgpts):
    c = tuple(int(v) for v in corners[0].ravel())
    for p, col in zip(imgpts, [(255,0,0),(0,255,0),(0,0,255)]):
        img = cv2.line(img, c, tuple(int(v) for v in p.ravel()), col, 5)
    return img

def draw_cube(img, imgpts):
    p = np.int32(imgpts).reshape(-1, 2)
    img = cv2.drawContours(img, [p[:4]], -1, (0,255,0), -3)
    for i, j in zip(range(4), range(4, 8)):
        img = cv2.line(img, tuple(p[i]), tuple(p[j]), (255,0,0), 3)
    return cv2.drawContours(img, [p[4:]], -1, (0,0,255), 3)

for fname in sorted(glob.glob('calib/*.jpg')):
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, PATTERN, None)
    if not ret: continue
    corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
    ok, rvec, tvec, inl = cv2.solvePnPRansac(objp, corners2, mtx, dist)
    pts, _ = cv2.projectPoints(axis, rvec, tvec, mtx, dist)
    cv2.imwrite(fname.replace('.jpg', '_axes.png'), draw_axes(img.copy(), corners2, pts))
    pts, _ = cv2.projectPoints(cube, rvec, tvec, mtx, dist)
    cv2.imwrite(fname.replace('.jpg', '_cube.png'), draw_cube(img.copy(), pts))
    print(fname, "distance mire-caméra (m):", float(np.linalg.norm(tvec)))

calib/IMG_0706.jpg distance mire-caméra (m): 0.28146440566980413
calib/IMG_0708.jpg distance mire-caméra (m): 0.24451332548469648
calib/IMG_0715.jpg distance mire-caméra (m): 0.26115823068421634
calib/IMG_0716.jpg distance mire-caméra (m): 0.3127075428538146
calib/IMG_0717.jpg distance mire-caméra (m): 0.3318924492716786
calib/IMG_0718.jpg distance mire-caméra (m): 0.23568093513146918
calib/IMG_0719.jpg distance mire-caméra (m): 0.19176712134991017
calib/IMG_0720.jpg distance mire-caméra (m): 0.30721844779229945
calib/IMG_0723.jpg distance mire-caméra (m): 0.33533532762851215
calib/image.jpg distance mire-caméra (m): 0.6645728576278723
calib/image2.jpg distance mire-caméra (m): 0.5027917751042648
